import csv
import re

regex: ^\d{4}\-(0?[1-9]|1[012])\-(0?[1-9]|[12][0-9]|3[01])$
for yyyy-mm-dd

what is mods, anyway.

look at regex 101, you can test out stuff to make sure your regex actually matches it

In [3]:
import csv
import re
import os

In [37]:
def generate_collection_list(ummabaskets):
    """
    Reads a CSV file and returns the data as a dictionary.

    Parameters:
    ummaa_filtered.csv (str): The path to the CSV file. Files are colocated.

    Returns:
    dict: A dictionary where each key is a column header and each value is a list of column values.
    """

    ummaa_data = list()

    with open('ummaa_filtered.csv', 'r', newline='', encoding='utf-8') as f:
        data = csv.DictReader(f)

        for row in data:
            row_dict = dict()
            for field in data.fieldnames:
                row_dict[field] = row[field]
            ummaa_data.append(row_dict)

        return ummaa_data

In [38]:

basket_list = generate_collection_list(ummabaskets)

In [39]:
basket_list[0:3]

[{'Object identifier': '8129',
  'Divisions': 'Ethnology & Material Culture',
  'Accession Number': 'Accession Number: 278',
  'Accession Description': 'Accession #278 Purchase 04/20/1926 (278)',
  'Quantity': 'Quantity: 1\nUnit: ea',
  'Type': 'Ethnographic',
  'Other Numbers': '',
  'Object Type': 'Basket',
  'Materials': 'Grass',
  'Verbatim Geography': 'Asia-Southeast Asia-Philippines-Cotabato-',
  'Geographic Location': '',
  'Political Location': '',
  'Culture': '',
  'Description': 'Basket with eight decorative legs. Red black natural color. Included note states "Basket made by women at Cottobatto"',
  'Curatorial Notes': ''},
 {'Object identifier': '8272',
  'Divisions': 'Ethnology & Material Culture',
  'Accession Number': 'Accession Number: 1',
  'Accession Description': 'Accession #1 UMMAA Expedition 1872 (1)',
  'Quantity': 'Quantity: 1\nUnit: ea',
  'Type': 'Ethnographic',
  'Other Numbers': '',
  'Object Type': 'Basket',
  'Materials': 'Fiber',
  'Verbatim Geography': 'A

In [40]:
basket_list[0]

{'Object identifier': '8129',
 'Divisions': 'Ethnology & Material Culture',
 'Accession Number': 'Accession Number: 278',
 'Accession Description': 'Accession #278 Purchase 04/20/1926 (278)',
 'Quantity': 'Quantity: 1\nUnit: ea',
 'Type': 'Ethnographic',
 'Other Numbers': '',
 'Object Type': 'Basket',
 'Materials': 'Grass',
 'Verbatim Geography': 'Asia-Southeast Asia-Philippines-Cotabato-',
 'Geographic Location': '',
 'Political Location': '',
 'Culture': '',
 'Description': 'Basket with eight decorative legs. Red black natural color. Included note states "Basket made by women at Cottobatto"',
 'Curatorial Notes': ''}

basket_list is a list of dictionaries. the keys are the fields we need to transform.

In [41]:
for key in basket_list[0]:
    print(key)

Object identifier
Divisions
Accession Number
Accession Description
Quantity
Type
Other Numbers
Object Type
Materials
Verbatim Geography
Geographic Location
Political Location
Culture
Description
Curatorial Notes


This code isolates the keys for one dictionary.

In [ ]:
for dicts in basket_list:
    print(dicts)

this code will list all of the keys for all of the dictionaries. I think that this will get us to where we need to go for the rest of our transform step. I think we could use "replace" to replace the keys with the dc term we chose to map it to.

In [ ]:
for dicts in basket_list:
    for key in dicts:
        print(key)

I checked with UMgpt and yes, we need a key mapping. 

In [24]:
key_map = {
 'Object identifier': 'dcterms:identifier',
 'Divisions': 'dcterms:isPartOf',
 'Accession Number': 'dcterms:identifier',
 'Accession Description': 'dcterms:provenance',
 'Quantity': 'dcterms:extent',
 'Type': 'dcterms:isPartOf',
 'Other Numbers': 'dcterms:alternative',
 'Object Type': 'dcterms:type',
 'Materials': 'dcterms:medium',
 'Display Date': 'dcterms:temporal',
 'Provenience': 'dcterms:provenance',
 'Verbatim Geography': 'dcterms:spatial',
 'Geographic Location': 'dcterms:spatial',
 'Political Location': 'dcterms:spatial',
 'Culture': 'dcterms:subject',
 'Description': 'dcterms:description',
 'Curatorial Notes': 'dcterms:description'
}

we removed the following from our MAP:
 'Original Number':

In [45]:
remapped_data = []

for dicts in basket_list:
    new_dicts = {}
    for old_key, value in dicts.items():
        new_key = key_map.get(old_key, old_key)
        new_dicts[new_key] = value
    remapped_data.append(new_dicts)

print(remapped_data[0])

{'dcterms:identifier': 'Accession Number: 278', 'dcterms:isPartOf': 'Ethnographic', 'dcterms:provenance': 'Accession #278 Purchase 04/20/1926 (278)', 'dcterms:extent': 'Quantity: 1\nUnit: ea', 'dcterms:alternative': '', 'dcterms:type': 'Basket', 'dcterms:medium': 'Grass', 'dcterms:spatial': '', 'dcterms:subject': '', 'dcterms:description': ''}


as you can see there are some issues with this code actually working. I am not sure why not! I think that we might need to clean it first and then make is a csv file so there aren't any odd surprises.

In [46]:
type(basket_list)

list

In [48]:
collection_set_list = 'transformed_ummaa_baskets.csv'
headers = ['dcterms:identifier','dcterms:isPartOf','dcterms:identifier','dcterms:provenance','dcterms:extent','dcterms:isPartOf','dcterms:alternative','dcterms:type','dcterms:medium','dcterms:temporal','dcterms:provenance','dcterms:spatial','dcterms:spatial','dcterms:spatial','dcterms:subject','dcterms:description','dcterms:description']

with open(collection_set_list, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in remapped_data:
        writer.writerow(item)